## PCE for Max Cut optimization problem

Pauli Correlation Encoding is an optimization algorithm that can tackle more than one binary variables per qubits assigned in the circuit. The number of binary variables depends on the order selected of the algorithm. Let's see some examples!

In [ ]:
from qarp.algorithms import PCE
import networkx as nx

In [ ]:
n_nodes = 100
n_edges = 300

from qarp import config

config.seed = 1234

G = nx.gnm_random_graph(n_nodes, n_edges, seed=config.seed)
for u, v in G.edges:
    G[u][v]['weight'] = 2

We first define the ansatz over which the computations are run. In this case, and following the seminar paper we will use Hardware Efficient Ansatz as a baseline, from OpenQARP:

We include a function to estimate the minimum number of qubits needed for performing the PCE approach with some given number of nodes and predefined order.

If merging is enabled, the number of qubits required is lower, because we are using correlators from different orders.

In [ ]:
from qarp.algorithms import calculate_qubits

order = 3
n_qubits = calculate_qubits(n_nodes, order, merging=True)
print("Number of qubits:", n_qubits)

In [ ]:
from qarp.blocks import HEABlock

he_wfn = HEABlock(n_qubits, 3, True, True, True, False).build()

By using the sampler primitive, we are able to perform meausurement reduction techniques during the computation. Thus, an improvement in the speedup is expected

In [ ]:
from qarp.algorithms import Sampler
from qarp.optimizers import ScipyOptimizer

pce = PCE(
    graph=G,
    order=order,
    ket=he_wfn,
    primitive=Sampler(),
    merging=True,
    initial_parameters=[0.1] * len(list(he_wfn.symbols)),
    verbose=True,
    optimizer=ScipyOptimizer("COBYQA"),
).build()
_, x, solution = pce.run()

On of the advantages of this algorithm is that the solution is computed in each iteration of the approach, so there's no need to sample again the initial ansatz. 

We can retrieve the solution from solution as follows:

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("PCE convergence")

iters = list(range(len(pce.history)))

ax1.plot(iters, [i[0] for i in pce.history])
ax1.set_ylabel("Loss")
ax1.set_xlabel("Iteration")
ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
ax1.yaxis.set_major_locator(MaxNLocator(nbins=6))
ax1.grid()

ax2.plot(iters, [i[1] for i in pce.history])
ax2.set_ylabel("MaxCut")
ax2.set_xlabel("Iteration")
ax2.xaxis.set_major_locator(MaxNLocator(integer=True))
ax2.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=True))
ax2.grid()

fig.tight_layout()
plt.show()

In [ ]:
from qarp.algorithms import classical_function_max_cut

print("f(x):", classical_function_max_cut(G, solution))
print("Binary string:", solution)

## Shot-based simution

Here we showcase the results by using TermwiseHadamardTest.

We need to increase the number of qubits, as we are not using merging=True, which was merging correlators from different orders

In [ ]:
from qarp.algorithms import calculate_qubits

order = 3
n_qubits = calculate_qubits(n_nodes, order, merging=False)
print("Number of qubits:", n_qubits)

In [ ]:
from qarp.blocks import HEABlock

he_wfn = HEABlock(n_qubits, 3, True, True, True, False).build()

In [ ]:
pce = PCE(
    graph=G,
    order=order,
    ket=he_wfn,
    primitive=Sampler(n_shots=1000),
    initial_parameters=[0.1] * len(list(he_wfn.symbols)),
    verbose=True,
    optimizer=ScipyOptimizer("COBYQA"),
).build()
_, x, solution = pce.run()

In [ ]:
print("Binary string:", solution)
print("f(x):", classical_function_max_cut(G, solution))

# Constraints

Constraints can now be handled with loop edges from and to the same node. 

In [ ]:
n_nodes = 50
order = 3
graph = nx.erdos_renyi_graph(n_nodes, .8)
n_qubits = calculate_qubits(n_nodes, order)
he_wfn = HEABlock(n_qubits, 3, True, True, True, True).build()

In [ ]:
n_constraints = 10
penalty = 5
for i in range(n_constraints):
    graph.add_edge(i, i, weight=penalty)

In [ ]:
pce = PCE(
    graph,
    order,
    he_wfn,
    primitive=Sampler(),
    initial_parameters=[0.1] * len(list(he_wfn.symbols)),
    verbose=True,
    optimizer=ScipyOptimizer("COBYQA"),
).build()
_, x, solution = pce.run()

In [ ]:
print("Binary string:", solution)
print("f(x):", classical_function_max_cut(graph, solution))